Download modelcode:

In [1]:
#!git clone https://github.com/eddie-euijun-hwang/SpaMo.git

Download checkpoint:

In [2]:
#!wget https://huggingface.co/chufakiks/SpaMo/resolve/main/spamo.ckpt

In [6]:
# Extract spatial features from a single video
!python vit_extract_feature.py \
    --video_path "ogvid.mp4"\
    --save_path "my_video_spatial.npy" \
    --model_name openai/clip-vit-base-patch32 \
    --s2_mode s2wrapping \
    --scales 1 2 \
    --batch_size 32 \
    --device cpu

Reading video from: ogvid.mp4
Loaded 504 frames
Initializing ViT model...
/home/lucia/.conda/envs/torch_legacy/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Extracting features...
Processing frames 0 to 32...
Processing frames 32 to 64...
Processing frames 64 to 96...
Processing frames 96 to 128...
Processing frames 128 to 160...
Processing frames 160 to 192...
Processing frames 192 to 224...
Processing frames 224 to 256...
Processing frames 256 to 288...
Processing frames 288 to 320...
Processing frames 320 to 352...
Processing frames 352 to 384...
Processing frames 384 to 416...
Processing frames 416 to 448...
Processing frames 448 to 480...
Processing fr

In [7]:
# Extract motion features from a single video
!python mae_extract_feature.py \
    --video_path "ogvid.mp4" \
    --save_path "my_video_motion.npy"\
    --model_name MCG-NJU/videomae-base \
    --overlap_size 8 \
    --batch_size 32 \
    --device cpu

Reading video from: ogvid.mp4
Loaded 504 frames
Applying sliding window (window_size=16, overlap=8)...
Created 62 chunks
Initializing VideoMAE model...
Extracting features...
Processing batch 1/2...
/home/lucia/.conda/envs/torch_legacy/lib/python3.10/site-packages/transformers/feature_extraction_utils.py:141: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /opt/conda/conda-bld/pytorch_1702400200538/work/torch/csrc/utils/tensor_new.cpp:261.)
  return torch.tensor(value)
Processing batch 2/2...
Feature shape: (62, 768)
Features saved to: my_video_motion.npy


In [3]:
import numpy as np
spatial = np.load('my_video_spatial.npy')
spatial.shape

(504, 1536)

In [ ]:
import torch
from omegaconf import OmegaConf
from utils.helpers import instantiate_from_config


In [ ]:
# Load config
config = OmegaConf.load("finetune.yaml")

# Instantiate model
model = instantiate_from_config(config.model)

# Load checkpoint
spatial_features = np.load('spatial_features.npy')
motion_features = np.load('motion_features.npy') 
ckpt_path = "spamo.ckpt"
state = torch.load(ckpt_path, map_location="cpu")
model.load_state_dict(state["state_dict"], strict=False)

FileNotFoundError: [Errno 2] No such file or directory: '/home/lucia/Documents/GitHub/SpaMoRun/finetune.yaml'

In [ ]:
# For a single video/sample
def predict_single(model, video_features, device):
    model.eval()
    model = model.to(device)
    
    with torch.no_grad():
        # Prepare input
        if isinstance(video_features, dict):
            inputs = {k: v.unsqueeze(0).to(device) if isinstance(v, torch.Tensor) else v 
                     for k, v in video_features.items()}
        else:
            inputs = video_features.unsqueeze(0).to(device)
        
        # Generate translation
        output = model.generate(inputs)
        
    return output['predictions'][0]

# Example usage
# video_feat = load_your_video_features()  # Your video features
# translation = predict_single(model, video_feat, device)
# print(f"Translation: {translation}")

Get spatial features

Get motion features

In [ ]:
from mae_extractsinglefeature import video_to_videomae_features
maefeat = video_to_videomae_features("ogvid.mp4")

/home/lucia/.conda/envs/torch_legacy/lib/python3.10/site-packages/transformers/feature_extraction_utils.py:141: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /opt/conda/conda-bld/pytorch_1702400200538/work/torch/csrc/utils/tensor_new.cpp:261.)
  return torch.tensor(value)


In [ ]:
video_feat = {'spatial': vitfeat, 'motion': maefeat}
predict_single(model, video_feat, 'cpu')